# MonoLM-346M: отдельная демонстрация инференса

Модель обучена с нуля на болгарском корпусе; checkpoint шага 5000. Этот ноутбук **не требует исходного репозитория, корпуса или локальных артефактов**. Он скачивает публичные веса и токенизатор с Hugging Face и генерирует продолжение текста.

Первая загрузка — около 1,38 ГБ. Подходит Python 3.12; можно использовать CPU или CUDA. Flash Attention и обучение не запускаются. Рекомендуется отдельное окружение или Colab. Это базовая модель, без chat template и SFT: качество связности и ответов ограничено.

[Модель](https://huggingface.co/eternl/monolm-346m-bg) · [Выполненный бенчмарк](evaluation.ipynb) · [Графики обучения](https://huggingface.co/spaces/eternl/monolm-training)

In [1]:
import importlib.metadata as metadata
import subprocess
import sys

required = {"transformers": "5.20.2", "tokenizers": "0.23.3", "huggingface-hub": "2.2.0", "safetensors": "0.8.0"}
installed = {d.metadata["Name"].lower().replace("_", "-"): d.version for d in metadata.distributions() if d.metadata.get("Name")}
packages = [f"{name}=={version}" for name, version in required.items() if installed.get(name) != version]
if "torch" not in installed or tuple(int(x) for x in installed["torch"].split(".")[:2]) < (2, 6):
    packages.append("torch>=2.6")
if packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *packages])
print("Зависимости готовы. Если эти библиотеки уже были импортированы до установки, перезапустите ядро.")

Зависимости готовы. Если эти библиотеки уже были импортированы до установки, перезапустите ядро.


In [2]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "eternl/monolm-346m-bg"
REVISION = "b75dfb5da41fe98b1fb669a878eb662389deb199"  # опубликованные веса шага 5000
CACHE_DIR = os.environ.get("MONOLM_DEMO_CACHE")
DEVICE = os.environ.get("MONOLM_DEMO_DEVICE", "cuda" if torch.cuda.is_available() else "cpu")
DTYPE = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE == "cuda" else torch.float32
torch.set_num_threads(min(8, os.cpu_count() or 1))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION, token=False, cache_dir=CACHE_DIR, local_files_only=False)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, token=False, cache_dir=CACHE_DIR,
    dtype=DTYPE, attn_implementation="sdpa",
).to(DEVICE).eval()
print("Устройство:", DEVICE, "· dtype:", DTYPE)
print("Параметров:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Контекст:", model.config.max_position_embeddings)
assert model.lm_head.weight is model.model.embed_tokens.weight

Loading weights:   0%|          | 0/354 [00:00<?, ?it/s]

Устройство: cpu · dtype: torch.float32
Параметров: 346,096,576
Контекст: 2048


In [3]:
def complete(prompt, max_new_tokens=64, temperature=0.8, seed=42):
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(DEVICE)
    length = inputs.input_ids.shape[1]
    if length + max_new_tokens > model.config.max_position_embeddings:
        raise ValueError("Промпт и продолжение должны помещаться в 2048 токенов")
    torch.manual_seed(seed)
    kwargs = {"do_sample": temperature > 0}
    if temperature > 0:
        kwargs.update(temperature=temperature, top_p=0.95, top_k=0)
    with torch.inference_mode():
        result = model.generate(
            **inputs, max_new_tokens=max_new_tokens, use_cache=True,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            suppress_tokens=[tokenizer.pad_token_id, tokenizer.unk_token_id, tokenizer.bos_token_id],
            **kwargs,
        )
    return tokenizer.decode(result[0, length:], skip_special_tokens=True)

Ниже два фиксированных промпта. Примеры не отбираются по удачности; это демонстрация загрузки и генерации. Измените `prompt` или вызовите `complete()` со своим началом болгарского текста. Для greedy decoding задайте `temperature=0`.

Полная оценка качества — в отдельном выполненном бенчмарке: там 100 заданий на модель, слепая рубрика и сравнение с Qwen.

In [4]:
for prompt in ["България е страна", "Момичето отвори прозореца и"]:
    print("Промпт:", prompt)
    print("Продолжение:", complete(prompt, max_new_tokens=32))
    print()

Промпт: България е страна


Продолжение:  с индустриална икономика, основана през 19 век. Икономическият растеж се развива бавно. През 20-ти век България се развива бързо. През следващите няколко години страната

Промпт: Момичето отвори прозореца и


Продолжение:  го зареди на свой ред в стаята си.
Той знаеше, че е започнал да става и че няма да бъде много щастлив. Помогна на полицията.

